Ejecuta primero la celda de **Utilidades**, luego la sección que necesites.

In [1]:

## Utilidades — Ejecutar siempre primero

import subprocess, sys, os, json as _j
from datetime import datetime
from IPython.display import display, Image, HTML

SEP1 = "─" * 55
SEP2 = "═" * 55
OUT  = "outputs"

def ejecutar_notebook(nombre, ruta):
    print("")
    print(SEP1)
    print("  Ejecutando: " + nombre)
    print("  Archivo   : " + ruta)
    print("  Inicio    : " + datetime.now().strftime("%H:%M:%S"))
    print(SEP1)
    if not os.path.exists(ruta):
        print("  ERROR: No se encontro " + ruta)
        return False
    r = subprocess.run(
        ["jupyter", "nbconvert", "--to", "notebook", "--execute",
         "--inplace", "--ExecutePreprocessor.timeout=600", ruta],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print("  OK Completado")
        return True
    print("  ERROR al ejecutar " + ruta)
    print(r.stderr[-400:] if r.stderr else "Sin detalle")
    return False

def ejecutar_etapas(etapas, modo):
    print("")
    print(SEP2)
    print("  PIPELINE: " + modo)
    print("  " + datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print(SEP2)
    resultados = [(n, ejecutar_notebook(n, r)) for n, r in etapas]
    print("")
    print(SEP2)
    print("  RESUMEN")
    print(SEP2)
    for n, ok in resultados:
        print("  [" + ("OK" if ok else "FAIL") + "]  " + n)
    print("")
    print("  " + str(sum(ok for _, ok in resultados)) + "/" + str(len(resultados)) + " etapas completadas")
    print(SEP2)
    print("")

def mostrar(json_file, plots):
    rp = os.path.join(OUT, json_file)
    if os.path.exists(rp):
        res = _j.load(open(rp, encoding='utf-8'))
        display(HTML("<h3>" + res.get('titulo','') + "</h3>"))
        for h in res.get('hipotesis', []):
            txt = "<b>" + h['id'] + ":</b> " + h['enunciado']
            if 'media_TI' in h:
                txt += ("<br>&nbsp;&nbsp;Media TI: " + str(h['media_TI']) +
                        " | No TI: " + str(h['media_no_TI']) +
                        " | <b>" + h.get('conclusion','') + "</b>")
            display(HTML(txt))
        for h in res.get('hallazgos', []):
            display(HTML("<b>" + h['seccion'] + "</b>: " + h['pregunta']))
        if 'kmeans' in res:
            km = res['kmeans']
            display(HTML("<b>K-Means k=" + str(km['k_optimo']) + "</b> — " +
                         "Inercia: " + str(km['inercia']) +
                         " | Silhouette: " + str(km['silhouette'])))
        if 'variable_Y' in res:
            y = res['variable_Y']
            display(HTML("<b>Variable Y:</b> <code>" + y['nombre'] + "</code> — " +
                         str(y.get('pct_positivos','')) + "% ofertas TI"))
    for plot in plots:
        p = os.path.join(OUT, plot)
        if os.path.exists(p):
            display(HTML("<b>" + plot + "</b>"))
            display(Image(filename=p, width=700))

print("Utilidades cargadas correctamente")


Utilidades cargadas correctamente


In [ ]:
## Solo Scraping
## Ejecuta main → carga datos en `Registros_Scraping`.

ejecutar_etapas([
    ("Scraping — Todos los integrantes", "main.ipynb"),
], "Solo Scraping")


═══════════════════════════════════════════════════════
  PIPELINE: Solo Scraping
  2026-06-10 19:45:48
═══════════════════════════════════════════════════════

───────────────────────────────────────────────────────
  Ejecutando: Scraping — Todos los integrantes
  Archivo   : main.ipynb
  Inicio    : 19:45:48
───────────────────────────────────────────────────────


In [ ]:
## Solo Limpieza
## Lee desde `raw_data`, limpia y guarda en `processed_data`.  
## Requisito: scraping previo.

ejecutar_etapas([
    ("Procesamiento y Limpieza",  "src/processor/processor.ipynb"),
    ("Validacion Contenedor A/B", "src/processor/verificar_separacion.ipynb"),
], "Solo Limpieza y Procesamiento")

In [ ]:
## Solo Análisis
## Ejecuta EDA + Clustering.  
## Requisito:limpieza previa.

ejecutar_etapas([
    ("EDA — Análisis Multivariado", "notebooks/eda_parte_A.ipynb"),
    ("EDA — Hipótesis",             "notebooks/eda_parte_B.ipynb"),
    ("Modelado No Supervisado",     "notebooks/NoModelado.ipynb"),
], "Solo Análisis (EDA + Clustering)")

In [ ]:
### Resultados — Análisis

mostrar('results_eda_A.json', [
    'plot_modalidad_por_categoria.png',
    'plot_horario_por_categoria.png',
    'plot_seniority_vs_modalidad.png',
    'plot_heatmap_correlacion_Y.png'
])
mostrar('results_eda_B.json', [
    'plot_h3_largo_descripcion.png',
    'plot_h4_horario_ti.png',
    'plot_resumen_hipotesis.png'
])
mostrar('results_nomodelado.json', [
    'plot_kmeans_codo.png',
    'plot_kmeans_pca.png',
    'plot_dbscan_pca.png'
])

In [ ]:
## Pipeline Completo
## Ejecuta todo en orden: Scraping → Limpieza → Análisis.

ejecutar_etapas([
    ("Scraping — Todos los integrantes", "main.ipynb"),
    ("Procesamiento y Limpieza",         "src/processor/processor.ipynb"),
    ("Validacion Contenedor A/B",        "src/processor/verificar_separacion.ipynb"),
    ("EDA — Análisis Multivariado",      "notebooks/eda_parte_A.ipynb"),
    ("EDA — Hipótesis",                  "notebooks/eda_parte_B.ipynb"),
    ("Modelado No Supervisado",          "notebooks/NoModelado.ipynb"),
], "Pipeline Completo")

In [ ]:
### Resultados — Análisis

mostrar('results_eda_A.json', [
    'plot_modalidad_por_categoria.png',
    'plot_horario_por_categoria.png',
    'plot_seniority_vs_modalidad.png',
    'plot_heatmap_correlacion_Y.png'
])
mostrar('results_eda_B.json', [
    'plot_h3_largo_descripcion.png',
    'plot_h4_horario_ti.png',
    'plot_resumen_hipotesis.png'
])
mostrar('results_nomodelado.json', [
    'plot_kmeans_codo.png',
    'plot_kmeans_pca.png',
    'plot_dbscan_pca.png'
])